# VERIDIC-DSA: Verifiable, Incentive-Compatible Dynamic Spectrum Access
## Google Colab Experiment Notebook

> **Paper:** VERIDIC-DSA — Verifiable Incentive-Driven Intelligent Cryptographic Dynamic Spectrum Access for 6G ISAC Networks  
> **Target:** IEEE JSAC, Special Issue on AI-Native 6G Wireless Networks

### ⚡ Recommended Runtime: GPU (T4 or A100)
Go to **Runtime → Change runtime type → Hardware accelerator → GPU**

---
### Contents
1. [Environment Setup](#setup)
2. [Configuration](#config)
3. [Quick Smoke Test (500 slots)](#smoke)
4. [Full VERIDIC-DSA Training](#full-train)
5. [Baseline Comparison](#baselines)
6. [Adversary Sweep](#adv-sweep)
7. [Non-Stationarity Stress Test](#nonstat)
8. [Result Analysis & Figures](#figures)
9. [Checkpoint Management](#checkpoints)

## 1. Environment Setup <a id='setup'></a>

In [ ]:
# Clone repository (skip if already cloned)
import os
if not os.path.exists('isac-experiment'):
    !git clone https://github.com/ispandey/isac-experiment.git
%cd isac-experiment

In [ ]:
# Run one-shot setup: installs deps, detects GPU, seeds RNG
exec(open('colab_setup.py').read())

In [ ]:
# Optional: Mount Google Drive for checkpoint persistence
# Comment out if you don't need persistent storage
from colab_setup import mount_drive
CHECKPOINT_DIR = mount_drive('veridic_checkpoints')
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'ppo_agent.pt')
print(f'Checkpoint path: {CHECKPOINT_PATH}')

## 2. Configuration <a id='config'></a>

In [ ]:
import src.config as cfg
import numpy as np
import torch

# ── Experiment parameters (edit here) ─────────────────────────────────────
K           = 10       # number of NCUs
ADV_FRAC    = 0.30     # adversary fraction
SEED        = 42
N_EPISODES  = 500      # increase to 50000 for full paper results
N_SLOTS_BL  = 2000     # slots for baseline runs

# Print key parameters
print('=== Simulation Parameters ===')
print(f'Carrier frequency : {cfg.FC_HZ/1e9:.0f} GHz')
print(f'Bandwidth         : {cfg.BW_HZ/1e6:.0f} MHz')
print(f'NCU count K       : {K}')
print(f'Adversary fraction: {ADV_FRAC:.0%}')
print(f'Noise power       : {cfg.NOISE_POWER_W:.2e} W')
print(f'CFAR Pfa          : {cfg.P_FA_ISAC}')
print(f'PPO device        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 3. Quick Smoke Test (500 slots) <a id='smoke'></a>

In [ ]:
from src.experiment import VeridicDSA
from src.adversary import AdversaryType

print('Running 500-slot smoke test ...')
sim = VeridicDSA(K=K, adversary_fraction=ADV_FRAC,
                 adversary_type=AdversaryType.SF, seed=SEED)

for i in range(500):
    sim.run_slot(i)

s = sim.metrics.summary()
print(f"\n=== Smoke Test Results (500 slots) ===")
print(f"  DAF  (detection accuracy)  : {s['DAF']:.4f}  (paper target ≥ 0.947)")
print(f"  HIP  (harmful interference): {s['HIP']:.6f} (paper target < 1e-3)")
print(f"  Revenue                    : {s['Revenue']:.2f}")
print(f"  Spectrum Efficiency (b/s/Hz): {s['SE']:.2f}")
print(f"  Honest fraction            : {s['f_honest']:.4f}")
print(f"  ZKP prover time (ms)       : {s['ZKP']['mean_prove_ms']:.1f}")

## 4. Full VERIDIC-DSA Training <a id='full-train'></a>

In [ ]:
from src.experiment import VeridicDSA
from src.adversary import AdversaryType
import json, os

sim = VeridicDSA(K=K, adversary_fraction=ADV_FRAC,
                 adversary_type=AdversaryType.SF, seed=SEED)

# Load existing checkpoint if available
if 'CHECKPOINT_PATH' in dir() and os.path.exists(CHECKPOINT_PATH):
    sim.ppo_agent.load_checkpoint(CHECKPOINT_PATH)
    print(f'Checkpoint loaded from {CHECKPOINT_PATH}')

summary, revenues = sim.run(
    n_episodes=N_EPISODES,
    verbose=True,
    log_every=50,
    checkpoint_path=CHECKPOINT_PATH if 'CHECKPOINT_PATH' in dir() else None,
    checkpoint_every=100,
)

print(f"\n=== Final Training Summary ===")
print(f"  DAF   : {summary['DAF']:.4f}")
print(f"  HIP   : {summary['HIP']:.6f}")
print(f"  Revenue (total) : {summary['Revenue']:.2f}")

# Save results
os.makedirs('results', exist_ok=True)
out = {'summary': summary, 'revenues': revenues}
with open(f'results/veridic_K{K}_s{SEED}.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
print('Results saved to results/')

## 5. Baseline Comparison <a id='baselines'></a>

In [ ]:
from src.baselines import BaselineSystem, BaselineType
from src.adversary import AdversaryType

bl_results = {}
for blt in BaselineType:
    bl = BaselineSystem(blt, K=K, adversary_fraction=ADV_FRAC,
                        adversary_type=AdversaryType.SF, seed=SEED)
    s = bl.run(n_slots=N_SLOTS_BL)
    bl_results[blt.name] = s
    print(f"  {blt.name}: DAF={s['DAF']:.4f}  HIP={s['HIP']:.6f}  Rev={s['Revenue']:.2f}")

# Compare with VERIDIC-DSA
if 'summary' in dir():
    print(f"\n  VERIDIC-DSA: DAF={summary['DAF']:.4f}  HIP={summary['HIP']:.6f}  Rev={summary['Revenue']:.2f}")

## 6. Adversary Fraction Sweep <a id='adv-sweep'></a>

In [ ]:
from src.experiment import VeridicDSA
from src.adversary import AdversaryType

ADV_FRACS = [0.0, 0.1, 0.2, 0.3, 0.33]
sweep_results = {}

for frac in ADV_FRACS:
    s_sim = VeridicDSA(K=K, adversary_fraction=frac,
                       adversary_type=AdversaryType.SF, seed=SEED)
    for i in range(N_SLOTS_BL):
        s_sim.run_slot(i)
    s = s_sim.metrics.summary()
    sweep_results[frac] = s
    print(f"  frac={frac:.2f}: DAF={s['DAF']:.4f}  HIP={s['HIP']:.6f}")

## 7. Non-Stationarity Stress Test <a id='nonstat'></a>

In [ ]:
from src.experiment import VeridicDSA
from src.adversary import AdversaryType

ns_sim = VeridicDSA(K=K, adversary_fraction=ADV_FRAC,
                    adversary_type=AdversaryType.SF, seed=SEED,
                    nonstationarity=True)
for i in range(2000):
    ns_sim.run_slot(i)
ns_s = ns_sim.metrics.summary()
print(f"Non-stationary run: DAF={ns_s['DAF']:.4f}  HIP={ns_s['HIP']:.6f}")

## 8. Result Analysis & Figures <a id='figures'></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Revenue convergence plot
if 'revenues' in dir() and revenues:
    window = 20
    rev_arr = np.array(revenues)
    smoothed = np.convolve(rev_arr, np.ones(window)/window, mode='valid')
    plt.figure(figsize=(10,4))
    plt.plot(rev_arr, alpha=0.3, label='Episode revenue')
    plt.plot(range(window-1, len(rev_arr)), smoothed, linewidth=2, label=f'{window}-ep moving avg')
    plt.xlabel('Episode'); plt.ylabel('Revenue'); plt.title('PPO Revenue Convergence')
    plt.legend(); plt.tight_layout()
    plt.savefig('results/figures/revenue_convergence_colab.png', dpi=150)
    plt.show()

# Adversary sweep plot
if 'sweep_results' in dir():
    fracs = sorted(sweep_results.keys())
    dafs  = [sweep_results[f]['DAF'] for f in fracs]
    hips  = [sweep_results[f]['HIP'] for f in fracs]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(fracs, dafs, 'o-'); axes[0].set_xlabel('Adversary fraction'); axes[0].set_ylabel('DAF')
    axes[0].set_title('DAF vs Adversary Fraction'); axes[0].grid(True)
    axes[1].semilogy(fracs, [max(h, 1e-9) for h in hips], 'o-r'); axes[1].set_xlabel('Adversary fraction')
    axes[1].set_ylabel('HIP'); axes[1].set_title('HIP vs Adversary Fraction'); axes[1].grid(True)
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/sweep_colab.png', dpi=150)
    plt.show()

In [ ]:
# Generate all 12 paper figures (requires pre-computed results)
!python analysis/generate_figures.py --n_slots 2000

## 9. Checkpoint Management <a id='checkpoints'></a>

In [ ]:
# Manually save checkpoint
if 'sim' in dir() and 'CHECKPOINT_PATH' in dir():
    sim.ppo_agent.save_checkpoint(CHECKPOINT_PATH)
    print(f'Checkpoint saved to {CHECKPOINT_PATH}')

# List saved checkpoints
import glob
for f in glob.glob('results/checkpoints/*.pt') + glob.glob('/content/drive/MyDrive/veridic_checkpoints/*.pt'):
    size = os.path.getsize(f) / 1024
    print(f'  {f}  ({size:.1f} KB)')